In [ ]:
#requirements
import re
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import spacy
nlp = spacy.load("en_core_web_sm")

import networkx as nx

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df = pd.read_csv("/content/train.csv", on_bad_lines='skip', engine='python')   # Kaggle CNN/DailyMail
df = df.sample(n=1000, random_state=42).reset_index(drop=True)

df = df[["id", "article", "highlights"]]

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          1000 non-null   object
 1   article     1000 non-null   object
 2   highlights  1000 non-null   object
dtypes: object(3)
memory usage: 23.6+ KB


In [ ]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)   # remove HTML tags
    text = re.sub(r"\s+", " ", text)     # normalize whitespace
    return text.strip()

df["article"] = df["article"].apply(clean_text)
df["highlights"] = df["highlights"].apply(clean_text)


In [ ]:
rows = []

for _, row in df.iterrows():
    article_id = row["id"]
    article = row["article"]
    highlights = row["highlights"]

    sentences = sent_tokenize(article)
    total_sentences = len(sentences)

    for idx, sent in enumerate(sentences):
        rows.append({
            "article_id": article_id,
            "sentence_index": idx,
            "sentence_text": sent,
            "total_sentences": total_sentences,
            "highlights": highlights
        })

sent_df = pd.DataFrame(rows)


In [ ]:
#download csv file
sent_df.to_csv("sentences_split.csv", index=False)


In [ ]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf.fit(
    sent_df["sentence_text"].tolist() +
    sent_df["highlights"].tolist()
)


TfidfVectorizer(stop_words='english')

In [ ]:
stop_words = set(stopwords.words("english"))

feature_rows = []

for article_id, group in sent_df.groupby("article_id"):

    article_text = " ".join(group["sentence_text"])
    highlight_text = group["highlights"].iloc[0]

    tfidf_article = tfidf.transform([article_text])
    tfidf_highlight = tfidf.transform([highlight_text])
    tfidf_sentences = tfidf.transform(group["sentence_text"])

    # TextRank
    sim_matrix = cosine_similarity(tfidf_sentences)
    nx_graph = nx.from_numpy_array(sim_matrix)
    textrank_scores = nx.pagerank(nx_graph)

    for i, row in group.iterrows():
        sent = row["sentence_text"]
        words = word_tokenize(sent)
        word_count = len(words)

        doc = nlp(sent)

        feature_rows.append({
            "article_id": article_id,

            # 1
            "sentence_index": row["sentence_index"],

            # 2
            "normalized_position": row["sentence_index"] / row["total_sentences"],

            # 3
            "sentence_length_words": word_count,

            # 4
            "tfidf_sim_article": cosine_similarity(
                tfidf_sentences[row.name - group.index[0]],
                tfidf_article
            )[0][0],

            # 5
            "tfidf_sim_highlights": cosine_similarity(
                tfidf_sentences[row.name - group.index[0]],
                tfidf_highlight
            )[0][0],

            # 6
            "ner_count": len(doc.ents),

            # 7
            "has_digit": int(any(char.isdigit() for char in sent)),

            # 8
            "stop_ratio": (
                sum(1 for w in words if w.lower() in stop_words) / word_count
                if word_count > 0 else 0
            ),

            # 9
            "textrank_score": textrank_scores[row.name - group.index[0]],

            "sentence_text": sent,
            "highlights": highlight_text
        })


In [ ]:
features_df = pd.DataFrame(feature_rows)


In [ ]:
features_df.describe()

,sentence_index,normalized_position,sentence_length_words,tfidf_sim_article,tfidf_sim_highlights,ner_count,has_digit,stop_ratio,textrank_score
count,37991.000000,37991.000000,37991.000000,37991.000000,37991.000000,37991.000000,37991.000000,37991.000000,37991.000000
mean,24.611513,0.486839,20.531810,0.234431,0.106263,1.767313,0.210997,0.346493,0.026322
std,21.156161,0.288655,12.464181,0.140821,0.131694,1.885740,0.408022,0.141370,0.017577
min,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000779
25%,9.000000,0.236842,12.000000,0.120172,0.000000,0.000000,0.000000,0.285714,0.014927
50%,20.000000,0.485714,19.000000,0.222379,0.060114,1.000000,0.000000,0.363636,0.022148
75%,34.000000,0.736842,28.000000,0.333283,0.169468,3.000000,0.000000,0.437500,0.032959
max,194.000000,0.994872,151.000000,0.785287,0.930073,44.000000,1.000000,0.833333,0.351052


In [ ]:
features_df.to_csv('features.csv',index=False)

In [ ]:
features_df["label"] = 0

for article_id, group in features_df.groupby("article_id"):
    k = len(sent_tokenize(group["highlights"].iloc[0]))

    top_k_idx = group["tfidf_sim_highlights"].nlargest(k).index
    features_df.loc[top_k_idx, "label"] = 1


In [ ]:
final_df = features_df[[
    "article_id",
    "sentence_index",
    "normalized_position",
    "sentence_length_words",
    "tfidf_sim_article",
    "ner_count",
    "has_digit",
    "stop_ratio",
    "textrank_score",
    "label"
]]

final_df.head()


,article_id,sentence_index,normalized_position,sentence_length_words,tfidf_sim_article,ner_count,has_digit,stop_ratio,textrank_score,label
0,002175ac42ef0c91b9fb7e07259413a8ee3979a3,0,0.00000,2,0.000000,0,0,0.500000,0.004815,0
1,002175ac42ef0c91b9fb7e07259413a8ee3979a3,1,0.03125,4,0.060328,1,0,0.000000,0.032103,0
2,002175ac42ef0c91b9fb7e07259413a8ee3979a3,2,0.06250,15,0.115041,0,0,0.400000,0.021460,0
3,002175ac42ef0c91b9fb7e07259413a8ee3979a3,3,0.09375,18,0.343291,0,0,0.444444,0.034641,0
4,002175ac42ef0c91b9fb7e07259413a8ee3979a3,4,0.12500,35,0.340579,7,0,0.342857,0.031594,0


In [ ]:
final_df.to_csv('final_dataset.csv',index=False)

Datasets:
1) sentence level spliting : sentences_split.csv
2) extracted features : features.csv
3) final data with labels : final_dataset.csv


In [ ]:
features_df.shape

(37991, 13)

In [ ]:
final_df.shape

(37991, 10)

In [ ]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37991 entries, 0 to 37990
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   article_id             37991 non-null  object 
 1   sentence_index         37991 non-null  int64  
 2   normalized_position    37991 non-null  float64
 3   sentence_length_words  37991 non-null  int64  
 4   tfidf_sim_article      37991 non-null  float64
 5   tfidf_sim_highlights   37991 non-null  float64
 6   ner_count              37991 non-null  int64  
 7   has_digit              37991 non-null  int64  
 8   stop_ratio             37991 non-null  float64
 9   textrank_score         37991 non-null  float64
 10  sentence_text          37991 non-null  object 
 11  highlights             37991 non-null  object 
 12  label                  37991 non-null  int64  
dtypes: float64(5), int64(5), object(3)
memory usage: 3.8+ MB


In [ ]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37991 entries, 0 to 37990
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   article_id             37991 non-null  object 
 1   sentence_index         37991 non-null  int64  
 2   normalized_position    37991 non-null  float64
 3   sentence_length_words  37991 non-null  int64  
 4   tfidf_sim_article      37991 non-null  float64
 5   ner_count              37991 non-null  int64  
 6   has_digit              37991 non-null  int64  
 7   stop_ratio             37991 non-null  float64
 8   textrank_score         37991 non-null  float64
 9   label                  37991 non-null  int64  
dtypes: float64(4), int64(5), object(1)
memory usage: 2.9+ MB
